In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENROUTER_API_KEY']:
    print("API Key is set")

API Key is set


In [3]:
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

response = llm.invoke("Hello")

print(response.content)

Hello! How can I assist you today?


In [5]:
res = llm.invoke("What is the capital of Rwanada")
print(res.content)


The capital of Rwanda is Kigali.


## **RAG IMPLEMENTATION WITH YOUR OWN DATA**

#### STEP - 1 : Preparing Document for your text

In [7]:
from langchain_core.documents import Document

my_text = """The tiger (Panthera tigris) is a large cat and a member of the genus Panthera native to Asia. It has a powerful, muscular body with a large head and paws, a long tail and orange fur with black, mostly vertical stripes. It is traditionally classified into nine recent subspecies, though some recognise only two subspecies, mainland Asian tigers and the island tigers of the Sunda Islands.

Throughout the tiger's range, it inhabits mainly forests, from coniferous and temperate broadleaf and mixed forests in the Russian Far East and Northeast China to tropical and subtropical moist broadleaf forests on the Indian subcontinent and Southeast Asia. The tiger is an apex predator and preys mainly on hooved mammals, which it takes by ambush. It lives a mostly solitary life and occupies home ranges, defending these from individuals of the same sex. The range of a male tiger overlaps with that of multiple females with whom he mates. Females give birth to usually two or three cubs that stay with their mother for about two years. When becoming independent, they leave their mother's home range and establish their own.

Since the early 20th century, tiger populations have lost at least 93% of their historic range and are locally extinct in West and Central Asia, in large areas of China and on the islands of Java and Bali. Today, the tiger's range is severely fragmented. It is listed as Endangered on the IUCN Red List of Threatened Species, as its range is thought to have declined by 53% to 68% since the late 1990s. Major threats to tigers are habitat destruction and fragmentation due to deforestation, poaching for fur and the illegal trade of body parts for medicinal purposes. Tigers are also victims of human–wildlife conflict as they attack and prey on livestock in areas where natural prey is scarce. The tiger is legally protected in all range countries. National conservation measures consist of action plans, anti-poaching patrols and schemes for monitoring tiger populations. In several range countries, wildlife corridors have been established and tiger reintroduction is planned."""

docs = [Document(page_content=my_text, metadata={"source":"Wikipedia", "documentID":"Doc1"})]

docs

[Document(metadata={'source': 'Wikipedia', 'documentID': 'Doc1'}, page_content="The tiger (Panthera tigris) is a large cat and a member of the genus Panthera native to Asia. It has a powerful, muscular body with a large head and paws, a long tail and orange fur with black, mostly vertical stripes. It is traditionally classified into nine recent subspecies, though some recognise only two subspecies, mainland Asian tigers and the island tigers of the Sunda Islands.\n\nThroughout the tiger's range, it inhabits mainly forests, from coniferous and temperate broadleaf and mixed forests in the Russian Far East and Northeast China to tropical and subtropical moist broadleaf forests on the Indian subcontinent and Southeast Asia. The tiger is an apex predator and preys mainly on hooved mammals, which it takes by ambush. It lives a mostly solitary life and occupies home ranges, defending these from individuals of the same sex. The range of a male tiger overlaps with that of multiple females with 

#### STEP - 2: Creating CHUNKS

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)
chunks


[Document(metadata={'source': 'Wikipedia', 'documentID': 'Doc1'}, page_content='The tiger (Panthera tigris) is a large cat and a member of the genus Panthera native to Asia. It has a powerful, muscular body with a large head and paws, a long tail and orange fur with black, mostly vertical stripes. It is traditionally classified into nine recent subspecies, though some recognise only two subspecies, mainland Asian tigers and the island tigers of the Sunda Islands.'),
 Document(metadata={'source': 'Wikipedia', 'documentID': 'Doc1'}, page_content="Throughout the tiger's range, it inhabits mainly forests, from coniferous and temperate broadleaf and mixed forests in the Russian Far East and Northeast China to tropical and subtropical moist broadleaf forests on the Indian subcontinent and Southeast Asia. The tiger is an apex predator and preys mainly on hooved mammals, which it takes by ambush. It lives a mostly solitary life and occupies home ranges, defending these from individuals of the 

### STEP -3 : Create Embeddings for the Chunks

In [15]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [17]:
embedding_model.embed_query("What is AI?")

[-0.01800537109375,
 0.01251220703125,
 -0.007022857666015625,
 0.00809478759765625,
 0.0162811279296875,
 -0.04058837890625,
 -0.0099334716796875,
 -0.0131378173828125,
 -0.04144287109375,
 0.03033447265625,
 -0.0002892017364501953,
 -0.047271728515625,
 -0.0224151611328125,
 -0.055755615234375,
 -0.01177215576171875,
 0.0036640167236328125,
 -0.0237579345703125,
 -0.017974853515625,
 0.0269317626953125,
 -0.0457763671875,
 0.00428009033203125,
 0.044921875,
 -0.01190185546875,
 0.002513885498046875,
 0.0205841064453125,
 -0.03973388671875,
 0.0360107421875,
 0.004520416259765625,
 -0.0264739990234375,
 -0.01427459716796875,
 0.03424072265625,
 -0.0279083251953125,
 0.0045318603515625,
 -0.0165557861328125,
 0.013214111328125,
 0.02587890625,
 -0.015655517578125,
 0.0113525390625,
 0.00276947021484375,
 -0.021514892578125,
 0.01457977294921875,
 0.005733489990234375,
 0.022857666015625,
 0.043975830078125,
 -0.050994873046875,
 0.01003265380859375,
 0.0007643699645996094,
 -0.02102661

### STEP - 4 : Storage

In [20]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [21]:
vector = []

for doc in chunks:
    vector = embedding_model.embed_documents([doc.page_content])
    vector.append(vector)

### STEP - 5: Semantic Search

In [25]:
vectorstore.similarity_search("What is Tiger's Scientific name?", k = 2
                              )

[Document(metadata={'source': 'Wikipedia', 'documentID': 'Doc1'}, page_content='The tiger (Panthera tigris) is a large cat and a member of the genus Panthera native to Asia. It has a powerful, muscular body with a large head and paws, a long tail and orange fur with black, mostly vertical stripes. It is traditionally classified into nine recent subspecies, though some recognise only two subspecies, mainland Asian tigers and the island tigers of the Sunda Islands.'),
 Document(metadata={'documentID': 'Doc1', 'source': 'Wikipedia'}, page_content="Throughout the tiger's range, it inhabits mainly forests, from coniferous and temperate broadleaf and mixed forests in the Russian Far East and Northeast China to tropical and subtropical moist broadleaf forests on the Indian subcontinent and Southeast Asia. The tiger is an apex predator and preys mainly on hooved mammals, which it takes by ambush. It lives a mostly solitary life and occupies home ranges, defending these from individuals of the 

####### ******TALK TO LLM**

In [30]:
context = vectorstore.similarity_search("What is Tiger?", k = 3)

In [32]:
response = llm.invoke(f"What is a Tiger? You can answer using the following context: {context}")

print(response.content)

A tiger (Panthera tigris) is a large cat native to Asia, recognized for its powerful, muscular build, large head and paws, long tail, and distinctive orange fur adorned with black vertical stripes. Traditionally, tigers are classified into nine recent subspecies, although some classifications recognize only two main groups: mainland Asian tigers and island tigers from the Sunda Islands.

Tigers primarily inhabit various forest types across their range, including coniferous, temperate broadleaf, and tropical forests. They are apex predators that primarily hunt hooved mammals by utilizing an ambush technique. Typically, tigers lead a solitary lifestyle and establish home ranges that they defend from other individuals of the same sex.

Tigers face significant threats, including habitat fragmentation from deforestation, poaching for their fur, and illegal trade of their body parts for medicinal purposes. They also often come into conflict with humans, particularly when they prey on livesto